In [1]:
# ============================================
# PHASE 4 — Composite Ranking vs Accuracy-Only Ranking (RQ4)
# ============================================

import pandas as pd
import numpy as np
import os

# ---- 1. LOAD PHASE 1 SUMMARY RESULTS FOR ALL THREE DATASETS ----
nslkdd = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\results\Phase1_NSLKDD_summary.csv", index_col=0)
unsw = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\results\Phase1_UNSWNB15_summary.csv", index_col=0)
cicids = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\results\Phase1_CICIDS2017_summary.csv", index_col=0)

datasets = {'NSL-KDD': nslkdd, 'UNSW-NB15': unsw, 'CICIDS2017': cicids}

# ---- 2. DEFINE COMPOSITE SCORE WEIGHTS ----
# MCC weighted highest (literature-justified primary metric, following Cantone et al.)
# F1 second (balances precision/recall, relevant for imbalanced SOC data)
# Accuracy, precision, recall given smaller, equal weights
weights = {
    'mcc': 0.35,
    'f1': 0.25,
    'precision': 0.15,
    'recall': 0.15,
    'accuracy': 0.10
}

print("Composite score weights:")
for metric, w in weights.items():
    print(f"  {metric}: {w}")

# ---- 3. CALCULATE COMPOSITE SCORE AND BOTH RANKINGS PER DATASET ----
composite_results = {}

for dataset_name, summary in datasets.items():
    df = summary.copy()
    df['composite_score'] = sum(df[metric] * weight for metric, weight in weights.items())

    df['accuracy_rank'] = df['accuracy'].rank(ascending=False)
    df['composite_rank'] = df['composite_score'].rank(ascending=False)

    composite_results[dataset_name] = df[['accuracy', 'mcc', 'f1', 'composite_score', 'accuracy_rank', 'composite_rank']]

    print(f"\n{'='*60}")
    print(f"{dataset_name}")
    print(f"{'='*60}")
    print(composite_results[dataset_name].sort_values('composite_rank').round(4))

# ---- 4. IDENTIFY WHERE RANKINGS DISAGREE ----
print(f"\n{'='*60}")
print("RANKING DISAGREEMENTS (Accuracy-only vs Composite)")
print(f"{'='*60}")

for dataset_name, df in composite_results.items():
    disagreements = df[df['accuracy_rank'] != df['composite_rank']]
    print(f"\n{dataset_name}:")
    if len(disagreements) == 0:
        print("  No disagreements — accuracy-only and composite rankings match exactly.")
    else:
        for algo in disagreements.index:
            acc_r = int(disagreements.loc[algo, 'accuracy_rank'])
            comp_r = int(disagreements.loc[algo, 'composite_rank'])
            print(f"  {algo}: accuracy-only rank {acc_r} -> composite rank {comp_r}")

# ---- 5. SAVE RESULTS ----
results_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\Phase4_results"
os.makedirs(results_folder, exist_ok=True)

for dataset_name, df in composite_results.items():
    df.round(4).to_csv(os.path.join(results_folder, f"Phase4_{dataset_name.replace('-', '')}_composite_ranking.csv"))

print(f"\n\nAll Phase 4 results saved successfully!")
print(results_folder)

Composite score weights:
  mcc: 0.35
  f1: 0.25
  precision: 0.15
  recall: 0.15
  accuracy: 0.1

NSL-KDD
                     accuracy     mcc      f1  composite_score  accuracy_rank  \
Random Forest          0.9989  0.9978  0.9988           0.9985            1.0   
Decision Tree          0.9980  0.9960  0.9978           0.9972            2.0   
KNN                    0.9958  0.9916  0.9955           0.9942            3.0   
SVM                    0.9916  0.9832  0.9910           0.9883            4.0   
Logistic Regression    0.9539  0.9073  0.9499           0.9354            5.0   

                     composite_rank  
Random Forest                   1.0  
Decision Tree                   2.0  
KNN                             3.0  
SVM                             4.0  
Logistic Regression             5.0  

UNSW-NB15
                     accuracy     mcc      f1  composite_score  accuracy_rank  \
Random Forest          0.9331  0.8664  0.9365           0.9117            1.0   
Decisi